# Particle Swarm Optimization Lab: Python companion
Interactive companion to Chapter 9 of [*Optimization Algorithms: AI techniques for design, planning, and control problems*](https://www.manning.com/books/optimization-algorithms) by Alaa Khamis (Manning, 2024).

These from-scratch implementations mirror the interactive simulations in the PSO Lab tutorial. They use only NumPy and Matplotlib, which are preinstalled in Google Colab. Change the function arguments to repeat the lab experiments.

## 1. Continuous PSO
Minimize the Michalewicz function (d = 2, minimum −1.8013 at (2.20, 1.57)) with the star, ring, Von Neumann and wheel topologies.

In [ ]:
# Continuous PSO from scratch (Chapter 9, Section 9.2)
# Minimizes the Michalewicz function with a selectable neighborhood topology.
import numpy as np
import matplotlib.pyplot as plt


def michalewicz(x, m=10):
    i = np.arange(1, x.shape[-1] + 1)
    return -np.sum(np.sin(x) * np.sin(i * x**2 / np.pi) ** (2 * m), axis=-1)


def neighbors(topology, n):
    """Indices each particle listens to (itself included)."""
    cols = int(np.ceil(np.sqrt(n)))
    nb = []
    for i in range(n):
        if topology == "star":            # gbest PSO: everyone talks to everyone
            s = range(n)
        elif topology == "ring":          # lbest PSO: left and right neighbors
            s = [i, (i - 1) % n, (i + 1) % n]
        elif topology == "von_neumann":   # grid: left, right, up, down
            s = [i, (i - 1) % n, (i + 1) % n, (i - cols) % n, (i + cols) % n]
        elif topology == "wheel":         # particle 0 is the hub
            s = range(n) if i == 0 else [i, 0]
        nb.append(np.unique(list(s)))
    return nb


def pso(f, bounds, n_particles=20, iters=100, w=0.792, c1=1.494, c2=1.494,
        topology="star", vmax_frac=0.2, seed=0):
    rng = np.random.default_rng(seed)
    lo, hi = map(np.asarray, bounds)
    d = lo.size
    x = rng.uniform(lo, hi, (n_particles, d))       # random positions
    v = np.zeros_like(x)                            # zero initial velocity
    vmax = vmax_frac * (hi - lo)
    pbest, pbest_f = x.copy(), f(x)                 # pbest = initial position
    nb = neighbors(topology, n_particles)
    history = [pbest_f.min()]
    for k in range(iters):
        # best position found inside each particle's neighborhood
        nbest = np.array([pbest[idx[np.argmin(pbest_f[idx])]] for idx in nb])
        r1, r2 = rng.random((2, n_particles, d))    # new numbers per dimension
        v = w * v + c1 * r1 * (pbest - x) + c2 * r2 * (nbest - x)   # eq. 9.1
        v = np.clip(v, -vmax, vmax)
        x = np.clip(x + v, lo, hi)                                   # eq. 9.2
        fx = f(x)
        improved = fx < pbest_f                                      # eq. 9.3
        pbest[improved], pbest_f[improved] = x[improved], fx[improved]
        history.append(pbest_f.min())
    g = np.argmin(pbest_f)
    return pbest[g], pbest_f[g], history


bounds = ([0, 0], [np.pi, np.pi])
for topology in ["star", "ring", "von_neumann", "wheel"]:
    best_x, best_f, hist = pso(michalewicz, bounds, topology=topology, seed=1)
    print(f"{topology:12s} x* = {np.round(best_x, 3)}  f* = {best_f:.4f}")
    plt.plot(hist, label=topology)

plt.axhline(-1.8013, ls="--", c="gray", label="known minimum")
plt.xlabel("iteration"); plt.ylabel("best cost"); plt.legend()
plt.title("Michalewicz (d = 2): effect of neighborhood topology")
plt.show()

## 2. Binary PSO
The worked bit update from the chapter (particle 4, bit 3), then a binary swarm that selects projects under a $60M budget (optimum value: 91).

In [ ]:
# Binary PSO from scratch (Chapter 9, Section 9.3)
import numpy as np
import matplotlib.pyplot as plt

sigmoid = lambda v: 1 / (1 + np.exp(-v))

# 1) The worked example: update bit 3 of particle 4 (100010)
v43, x43, pbest43, gbest3, phi1, phi2, r43 = 0.23, 0, 1, 1, 1.5, 1.9, 0.6
v_new = v43 + phi1 * (pbest43 - x43) + phi2 * (gbest3 - x43)          # eq. 9.8
prob = sigmoid(v_new)                                                  # eq. 9.9
x_new = 1 if prob > r43 else 0                                         # eq. 9.10
print(f"v = {v_new:.2f}, sig(v) = {prob:.3f}, new bit = {x_new}")    # 3.63, 0.974, 1

# 2) A full BPSO run: select projects under a budget (0/1 knapsack)
costs  = np.array([12, 7, 15, 5, 9, 6, 8, 11, 4, 7, 13, 14, 3, 4, 10, 5])
values = np.array([18, 11, 21, 9, 12, 10, 11, 16, 6, 9, 17, 20, 5, 5, 13, 7])
budget = 60

def fitness(X):  # maximize value, penalize budget overruns
    over = np.maximum(0, X @ costs - budget)
    return X @ values - 3 * over


def bpso(n_particles=20, iters=100, w=1.0, c1=2.0, c2=2.0, vmax=4.0, seed=0):
    rng = np.random.default_rng(seed)
    d = costs.size
    x = rng.integers(0, 2, (n_particles, d))
    v = np.zeros((n_particles, d))
    pbest, pbest_f = x.copy(), fitness(x)
    history = [pbest_f.max()]
    for k in range(iters):
        g = pbest[np.argmax(pbest_f)]                       # star topology
        r1, r2 = rng.random((2, n_particles, d))
        v = w * v + c1 * r1 * (pbest - x) + c2 * r2 * (g - x)
        v = np.clip(v, -vmax, vmax)                         # keeps sig(v) off 0 and 1
        x = (rng.random((n_particles, d)) < sigmoid(v)).astype(int)
        fx = fitness(x)
        better = fx > pbest_f
        pbest[better], pbest_f[better] = x[better], fx[better]
        history.append(pbest_f.max())
    best = pbest[np.argmax(pbest_f)]
    return best, history


best, hist = bpso(seed=3)
print("selected projects:", np.flatnonzero(best) + 1)
print("cost =", best @ costs, "/", budget, " value =", best @ values)
plt.plot(hist); plt.xlabel("iteration"); plt.ylabel("best value")
plt.title("Binary PSO on project selection"); plt.show()

## 3. Permutation-based PSO
Swap-based subtraction, addition and multiplication (Figures 9.8 to 9.10), then the 20-city US TSP from Listing 9.2. Set `hybrid=True` to add 2-opt local search.

In [ ]:
# Permutation-based PSO from scratch (Chapter 9, Sections 9.4 and 9.6)
# Velocity = list of swaps. Subtraction, addition and multiplication are redefined.
import math
import numpy as np
import matplotlib.pyplot as plt


def subtract(target, x):
    """target - x: the swaps that turn x into target (eq. 9.1 differences)."""
    x, swaps = list(x), []
    for i, c in enumerate(target):
        if x[i] != c:
            swaps.append((c, x[i]))
            j = x.index(c)
            x[i], x[j] = x[j], x[i]
    return swaps


def add(x, velocity):
    """x + v: apply each swap (exchange the two cities) in order."""
    x = list(x)
    for a, b in velocity:
        i, j = x.index(a), x.index(b)
        x[i], x[j] = x[j], x[i]
    return x


def multiply(c, velocity):
    """c * v: c = 0 empties, c < 1 truncates, c > 1 augments from the top."""
    n = len(velocity)
    if c <= 0 or n == 0:
        return []
    k = math.ceil(c * n) if c < 1 else math.floor(c * n)
    return [velocity[i % n] for i in range(k)]


# The book's figures 9.8 to 9.10
print(subtract([2, 4, 6, 1, 5, 3, 8, 7], [6, 4, 7, 1, 5, 3, 8, 2]))  # [(2, 6), (6, 7)]
print(add([2, 4, 6, 1, 5, 3, 8, 7], [(1, 3), (4, 7)]))              # [2, 7, 6, 3, 5, 1, 8, 4]
print(multiply(0.4, [(2, 6), (3, 7)]), multiply(1.8, [(2, 6), (3, 7)]))

# TSP: the 20 US cities of Listing 9.2
cities = {
    "New York City": (40.72, -74.00), "Philadelphia": (39.95, -75.17),
    "Baltimore": (39.28, -76.62), "Charlotte": (35.23, -80.85),
    "Memphis": (35.12, -89.97), "Jacksonville": (30.32, -81.70),
    "Houston": (29.77, -95.38), "Austin": (30.27, -97.77),
    "San Antonio": (29.53, -98.47), "Fort Worth": (32.75, -97.33),
    "Dallas": (32.78, -96.80), "San Diego": (32.78, -117.15),
    "Los Angeles": (34.05, -118.25), "San Jose": (37.30, -121.87),
    "San Francisco": (37.78, -122.42), "Indianapolis": (39.78, -86.15),
    "Phoenix": (33.45, -112.07), "Columbus": (39.98, -82.98),
    "Chicago": (41.88, -87.63), "Detroit": (42.33, -83.05),
}
names = list(cities)
lat, lon = np.radians(np.array(list(cities.values()))).T


def haversine_km(i, j):
    a = (np.sin((lat[j] - lat[i]) / 2) ** 2
         + np.cos(lat[i]) * np.cos(lat[j]) * np.sin((lon[j] - lon[i]) / 2) ** 2)
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))


n = len(names)
D = np.array([[haversine_km(i, j) for j in range(n)] for i in range(n)])
tour_length = lambda t: sum(D[t[k], t[(k + 1) % n]] for k in range(n))


def two_opt(t):
    """Optional local search: reverse segments while the tour gets shorter."""
    t, improved = t[:], True
    while improved:
        improved = False
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                a, b, c, d = t[i - 1], t[i], t[j], t[(j + 1) % n]
                if D[a, c] + D[b, d] < D[a, b] + D[c, d] - 1e-9:
                    t[i:j + 1] = t[i:j + 1][::-1]
                    improved = True
    return t


def permutation_pso(n_particles=40, iters=300, w=0.3, c1=1.5, c2=1.5,
                    vmax=10, hybrid=False, seed=0):
    rng = np.random.default_rng(seed)
    x = [list(rng.permutation(n)) for _ in range(n_particles)]
    v = [[] for _ in range(n_particles)]
    pbest = [p[:] for p in x]
    pbest_f = [tour_length(p) for p in x]
    history = [min(pbest_f)]
    for k in range(iters):
        g = pbest[int(np.argmin(pbest_f))]                   # star topology
        for i in range(n_particles):
            r1, r2 = rng.random(2)
            v[i] = (multiply(w, v[i])
                    + multiply(c1 * r1, subtract(pbest[i], x[i]))
                    + multiply(c2 * r2, subtract(g, x[i])))[:vmax]
            x[i] = add(x[i], v[i])
            f = tour_length(x[i])
            if f < pbest_f[i]:
                pbest[i], pbest_f[i] = x[i][:], f
        if hybrid:                                            # PSO + 2-opt on the best
            b = int(np.argmin(pbest_f))
            pbest[b] = two_opt(pbest[b])
            pbest_f[b] = tour_length(pbest[b])
        history.append(min(pbest_f))
    b = int(np.argmin(pbest_f))
    return pbest[b], pbest_f[b], history


best, length, hist = permutation_pso(seed=0)               # try hybrid=True
start = best.index(0)                                     # start at New York City
best = best[start:] + best[:start]
print(" -> ".join(names[c] for c in best))
print(f"Route length: {length:.1f} km")
plt.plot(hist); plt.xlabel("iteration"); plt.ylabel("best tour length (km)")
plt.title("Permutation PSO on the 20-city TSP"); plt.show()

## 4. Adaptive PSO
Inertia schedules (RS, LTV, NLTV; eqs. 9.11 to 9.13) and time-varying c1, c2 (eqs. 9.14, 9.15), then a comparison of the strategies on the 10-dimensional Rastrigin function.

In [ ]:
# Adaptive PSO from scratch (Chapter 9, Section 9.5)
# Inertia schedules (eqs. 9.11 to 9.13) and time-varying c1, c2 (eqs. 9.14, 9.15).
import numpy as np
import matplotlib.pyplot as plt

W_MAX, W_MIN = 0.9, 0.4


def inertia(strategy, t, t_max, w_prev, rng):
    if strategy == "RS":      # random selection, mean 0.75 (eq. 9.11)
        return 0.5 + rng.random() / 2
    if strategy == "LTV":     # linear time varying (eq. 9.12)
        return (W_MAX - W_MIN) * (t_max - t) / t_max + W_MIN
    if strategy == "NLTV":    # nonlinear time varying, w_0 = 0.9 (eq. 9.13)
        if t == 0:
            return W_MAX
        return (w_prev - W_MIN) * (t_max - (t - 1)) / (t_max + W_MIN) + W_MIN
    return 0.792              # constant inertia


def coefficients(t, t_max, c1=(0.5, 2.0), c2=(0.8, 2.5)):
    """c1 falls from c1_max to c1_min, c2 rises from c2_min to c2_max."""
    c1_t = (c1[1] - c1[0]) * (t_max - t) / t_max + c1[0]     # eq. 9.14
    c2_t = c2[1] - (c2[1] - c2[0]) * (t_max - t) / t_max     # eq. 9.15
    return c1_t, c2_t


def rastrigin(x):
    return 10 * x.shape[-1] + np.sum(x**2 - 10 * np.cos(2 * np.pi * x), axis=-1)


def adaptive_pso(f, lo, hi, d=10, n=20, t_max=300, strategy="LTV",
                 tvac=False, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(lo, hi, (n, d))
    v = np.zeros_like(x)
    vmax = 0.2 * (hi - lo)
    pbest, pbest_f = x.copy(), f(x)
    w, history = None, [pbest_f.min()]
    for t in range(t_max):
        w = inertia(strategy, t, t_max, w, rng)
        c1, c2 = coefficients(t, t_max) if tvac else (1.49, 1.49)
        g = pbest[np.argmin(pbest_f)]
        r1, r2 = rng.random((2, n, d))
        v = np.clip(w * v + c1 * r1 * (pbest - x) + c2 * r2 * (g - x), -vmax, vmax)
        x = np.clip(x + v, lo, hi)
        fx = f(x)
        better = fx < pbest_f
        pbest[better], pbest_f[better] = x[better], fx[better]
        history.append(pbest_f.min())
    return pbest_f.min(), history


# Figures 9.12 and 9.13: the schedules themselves
t_max, rng = 100, np.random.default_rng(1)
w_rs = [inertia("RS", t, t_max, None, rng) for t in range(t_max + 1)]
w_ltv = [inertia("LTV", t, t_max, None, rng) for t in range(t_max + 1)]
w_nltv, w = [], None
for t in range(t_max + 1):
    w = inertia("NLTV", t, t_max, w, rng)
    w_nltv.append(w)
c1c2 = np.array([coefficients(t, t_max) for t in range(t_max + 1)])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(w_rs, "g--", label="RS"); ax[0].plot(w_ltv, "r", label="LTV")
ax[0].plot(w_nltv, "b:", label="NLTV"); ax[0].set_title("Inertia weight")
ax[1].plot(c1c2[:, 0], "r", label="c1(t)"); ax[1].plot(c1c2[:, 1], "b--", label="c2(t)")
ax[1].set_title("Acceleration coefficients")
for a in ax:
    a.set_xlabel("iteration"); a.legend()
plt.show()

# Compare strategies on 10-dimensional Rastrigin (20 runs each)
strategies = [("Constant w", "const", False), ("RS", "RS", False), ("LTV", "LTV", False),
              ("NLTV", "NLTV", False), ("LTV + varying c1, c2", "LTV", True)]
for name, s, tv in strategies:
    runs = [adaptive_pso(rastrigin, -5.12, 5.12, strategy=s, tvac=tv, seed=k)
            for k in range(20)]
    finals = np.array([r[0] for r in runs])
    print(f"{name:22s} mean = {finals.mean():7.3f}  std = {finals.std():6.3f}  best = {finals.min():.3f}")
    plt.plot(np.mean([r[1] for r in runs], axis=0), label=name)
plt.yscale("log"); plt.xlabel("iteration"); plt.ylabel("mean best cost")
plt.title("Adaptive PSO on Rastrigin (d = 10)"); plt.legend(); plt.show()